In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

In [ ]:
insurance = pd.read_csv("insurance.csv")

insurance.head()

In [ ]:
fig = px.scatter(
    insurance,
    x="bmi",
    y="charges",
    color="smoker",
    trendline="ols",
    opacity=0.65,
    color_discrete_map={
        "no": "#232D4B",
        "yes": "#E57200"
    },
    title="BMI Versus Insurance Charges by Smoking Status",
    labels={
        "bmi": "Body Mass Index (BMI)",
        "charges": "Insurance Charges ($)",
        "smoker": "Smoker"
    }
)

fig.show()

In [ ]:
y = insurance["charges"]

numeric_features = [
    "age",
    "bmi",
    "children"
]

categorical_features = [
    "sex",
    "smoker",
    "region"
]

X = insurance[
    numeric_features + categorical_features
]

X.head()

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            "passthrough",
            numeric_features
        ),
        (
            "categorical",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ],
    verbose_feature_names_out=False
)

In [ ]:
X_transformed = preprocessor.fit_transform(X)

X_transformed = pd.DataFrame(
    X_transformed,
    columns=preprocessor.get_feature_names_out(),
    index=X.index
)

X_transformed.head()

In [ ]:
ols_pipeline = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "model",
            LinearRegression()
        )
    ]
)

ols_pipeline.fit(X, y)

In [ ]:
y_hat = ols_pipeline.predict(X)

In [ ]:
model_results = X.copy()

model_results["observed_charges"] = y

model_results["predicted_charges"] = y_hat

model_results["residual"] = (
    model_results["observed_charges"]
    -
    model_results["predicted_charges"]
)

model_results.head()

In [ ]:
fig = px.scatter(
    model_results,
    x="predicted_charges",
    y="residual",
    color="smoker",
    opacity=0.65,
    title="Residuals Versus Predicted Insurance Charges",
    labels={
        "predicted_charges": "Predicted Charges ($)",
        "residual": "Residual: Observed − Predicted ($)",
        "smoker": "Smoker"
    },
    color_discrete_map={
        "no": "#232D4B",
        "yes": "#E57200"
    }
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="black"
)

fig.show()

In [ ]:
feature_names = (
    ols_pipeline
    .named_steps["preprocessing"]
    .get_feature_names_out()
)


In [ ]:
coefficient_table = pd.DataFrame({
    "predictor": feature_names,
    "coefficient": (
        ols_pipeline
        .named_steps["model"]
        .coef_
    )
})

intercept = (
    ols_pipeline
    .named_steps["model"]
    .intercept_
)

print(f"Intercept: ${intercept:,.2f}")

coefficient_table

In [ ]:
from sklearn.metrics import r2_score

In [ ]:
r_squared = r2_score(
    y,
    y_hat
)

n = len(y)

p = len(feature_names)

adjusted_r_squared = (
    1
    -
    (1 - r_squared)
    *
    (n - 1)
    /
    (n - p - 1)
)

print(f"R²:          {r_squared:.3f}")
print(f"Adjusted R²: {adjusted_r_squared:.3f}")
print(f"Number of observations: {n}")
print(f"Number of predictors after encoding: {p}")